# TabDDPM

Last of the four generation notebooks, covering the most recent method, following the same
shape as the previous three.

This notebook produces two datasets rather than one. The first uses the library defaults
and is saved as the default dataset. The second uses the configuration chosen by the search
below and is saved as the tuned dataset. Both are the size of the training set, and each is
recorded in the shared timing log under its own name, with its settings and the hardware it
ran on.

Those two datasets are the project's two experiments. The first asks how the four methods
compare out of the box, which is what a practitioner would get by installing a library and
following its documentation. The second asks whether that comparison survives tuning, so
that a method is not judged solely on how well its authors chose its defaults. Running this
notebook from top to bottom produces both.

TabDDPM is a diffusion model. During training it repeatedly takes a real record, corrupts
it with noise in many small steps until nothing but noise remains, and learns to undo each
step. To generate a record it runs that process in reverse, starting from pure noise and
gradually shaping it into something realistic. The idea powers modern image generators, and
TabDDPM adapts it to tables that mix numbers and categories.

It is included as the recent state-of-the-art method alongside the three established
baselines. Its expected trade-off is computational cost, since generating each record means
running many denoising steps, so both training and generation are slower than the other
methods.

It comes from a different library than the sdv-based methods, which is why the setup below
is more involved and why the session has to restart part way through.

## Setup

synthcity is a large library with many dependencies, so this install takes several minutes
on a fresh session. It only needs to happen once.

Three pinning problems have to be handled together, all caused by the same thing. synthcity
constrains its dependencies to older versions than the environment ships, and pip downgrades
some packages while leaving others untouched, so the set stops matching itself.

1. opacus is pinned because the newer release refers to a part of PyTorch that is not
   present in the version synthcity installs, which would stop the library loading.
2. torch and torchvision have to be installed as a matched pair. synthcity requires torch
   below 2.3, so pip downgrades PyTorch but leaves torchvision, which was compiled against
   the newer one. The import then fails, because torchvision's compiled operators no longer
   match the PyTorch they are registering against.
3. numpy, pandas and pyarrow have to match too. synthcity requires numpy below version 2,
   and the preinstalled pandas and pyarrow are built against numpy 2, so they fail with a
   binary incompatibility error once numpy is downgraded.

The session restart is not optional, and the cell does it automatically. Installing new
versions only replaces the files on disk; the kernel still holds the previous torch and
numpy in memory, and a compiled extension cannot bind to an already imported mismatched
library. Continuing without restarting produces misleading errors that look like a different
problem but are the same one.

So the message about the session restarting is expected rather than a failure. Once it comes
back, skip this cell. Restarting costs nothing, because the training data is read back from
disk rather than held in memory.

In [ ]:
# torch and torchvision are pinned as a matched pair. synthcity requires torch<2.3,
# so installing it alone downgrades PyTorch but leaves the installed torchvision, which was
# compiled against the newer PyTorch. The import then fails with
# "RuntimeError: operator torchvision::nms does not exist". Installing them together
# in one resolution pass keeps the compiled operators consistent.
%pip install -q synthcity "opacus==1.5.2" "torch==2.2.2" "torchvision==0.17.2"

# synthcity also requires numpy below version 2, and The preinstalled pandas and pyarrow are
# compiled against numpy 2, so they fail with "ValueError: numpy.dtype size changed"
# once numpy is downgraded. These pins are from the numpy 1.x generation.
%pip install -q "numpy==1.26.4" "pandas==2.1.4" "pyarrow==15.0.2"

# The restart is not optional and cannot be done later. pip has replaced torch and
# numpy on disk, but this kernel still holds the previous versions in memory, and a
# compiled extension cannot bind to an already-imported mismatched library. Skipping
# it produces confusing failures such as "partially initialized module 'torchvision'
# has no attribute 'extension'". Rather than rely on the restart being done by hand,
# this cell performs it itself.
import sys

print(
    "\n"
    "==================================================================\n"
    "  Install finished. RESTARTING THE RUNTIME NOW automatically.\n"
    "\n"
    "  The message about the session restarting is expected, not an\n"
    "  error. When it comes back, skip this cell and continue from\n"
    "  the working folder cell onward.\n"
    "\n"
    "  The training data is read back from disk, so the earlier\n"
    "  steps do not need repeating.\n"
    "==================================================================\n",
    flush=True,
)
sys.stdout.flush()

try:
    from IPython import get_ipython

    get_ipython().kernel.do_shutdown(restart=True)
except Exception:
    print("Automatic restart failed. Restart the session manually.")

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.3 which is incompatible.
spopt 0.7.0 requires networkx>=3.2, but you have networkx 2.8.8 which is incompatible.
momepy 0.11.0 requires networkx>=3.2, but you have networkx 2.8.8 which is incompatible.
mapclassify 2.10.0 requires networkx>=3.2, but you have networkx 2.8.8 which is incompatible.
tobler 0.14.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
xarray-einstats 0.10.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
xgbse 0.3.3 requires pandas<3.0.0,>=2.2.0, but you have pandas 2.1.4 which is incompatible.
google-colab 1.0.0 

: 

## Working folder

Sets the project folder so everything the pipeline writes, the cohort, the synthetic
datasets, the outputs and the figures, persists between sessions rather than sitting on
temporary storage.

The cohort and the synthetic datasets derive from MIMIC-IV, which is credentialed data
under a PhysioNet data use agreement. Keep the folder private, do not share it, and
delete the data once the work is finished.

In [1]:
import os
from pathlib import Path

# Use the shared project folder when one is available, otherwise stay in the
# current directory.
try:
    from google.colab import drive
    drive.mount("/content/drive")
    project_dir = Path("/content/drive/MyDrive/mimic-synthetic-pipeline")
    project_dir.mkdir(parents=True, exist_ok=True)
    os.chdir(project_dir)
    print(f"Working folder: {project_dir}")
except ImportError:
    print("Using the local working folder.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Working folder: /content/drive/MyDrive/mimic-synthetic-pipeline


## Hardware check

Hardware matters most for this notebook. Diffusion models are the heaviest method in the
comparison, and on a CPU this run would take hours rather than minutes.

In [2]:
from importlib.metadata import PackageNotFoundError, version

import torch

# Read torchvision's version from its package metadata rather than importing it.
# Importing a mismatched torchvision is itself the failure mode: its compiled
# extension binds to torch at import time, so it hangs or raises rather than
# reporting anything useful. Reading the metadata checks the pairing without
# triggering it.
try:
    torchvision_version = version("torchvision")
except PackageNotFoundError:
    torchvision_version = "not installed"

print(f"torch {torch.__version__}, torchvision {torchvision_version}")
if not (torch.__version__.startswith("2.2") and torchvision_version.startswith("0.17")):
    print(
        "\nWARNING: expected torch 2.2.x paired with torchvision 0.17.x. This pair is\n"
        "mismatched, and synthcity imports torchvision, so it will fail. Delete the\n"
        "runtime and start a fresh session with a GPU, then\n"
        "run the setup cell at the top before anything else."
    )

cuda_available = torch.cuda.is_available()
print(f"CUDA available: {cuda_available}")
if cuda_available:
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print(
        "No GPU detected. TabDDPM will train on CPU, which can take hours on this cohort. "
        "If a GPU was expected, enable it in the session settings, then restart "
        "the session and re-run from the beginning."
    )

torch 2.2.2+cu121, torchvision 0.17.2
CUDA available: True
GPU: NVIDIA A100-SXM4-40GB


## Training data

The version check confirms the pins from the setup cell actually took effect, since a
mismatch surfaces later as an opaque binary incompatibility error rather than anything
readable.

In [3]:
from pathlib import Path

import numpy as np
import pandas as pd

# Confirm the install pins took effect. numpy must be below 2 for synthcity, and
# pandas must match it. A mismatch shows up as "ValueError: numpy.dtype size changed".
print(f"numpy {np.__version__}, pandas {pd.__version__}")
if int(np.__version__.split(".")[0]) >= 2:
    raise RuntimeError(
        f"numpy {np.__version__} is too new for synthcity, which requires numpy<2. "
        "Run the setup cell at the top, then restart the session."
    )

if not Path("data/train.parquet").exists():
    raise FileNotFoundError(
        "data/train.parquet not found. Run the extraction and preparation steps first."
    )

TARGET = "readmitted_30d"
train_df = pd.read_parquet("data/train.parquet")

# Guard against pandas nullable Int64 columns (from an older saved train.parquet)
# that some generator libraries cannot read. Coerce them to plain numpy int64.
for _c in train_df.columns:
    if str(train_df[_c].dtype) == "Int64":
        train_df[_c] = train_df[_c].astype("int64")
print(f"Training data: {len(train_df):,} admissions, readmission rate {train_df[TARGET].mean():.4f}")

numpy 1.26.4, pandas 2.1.4
Training data: 428,143 admissions, readmission rate 0.2059


## Diagnostic: which continuous encoder lets the numeric columns train

A diagnostic rather than part of the pipeline proper. It has already been run and its
conclusion is recorded below, so it can be skipped. It is kept because it is the evidence
behind the non-default continuous_encoder setting used further down.

### The problem

A full 1000-epoch run converged, with the loss flat to within 0.03% across the final fifth
of training, but its numeric loss sat at exactly 1.0000 and never moved. For a diffusion
model trained to predict Gaussian noise under a mean-squared-error loss, predicting zero
gives a loss of exactly 1, so that value is the signature of a branch that has learned
nothing at all. The categorical branch trained normally over the same run, so the failure
was specific to the continuous columns and was not a matter of training length.

The consequence was visible in the generated data. Without denoising, variance compounds
through the reverse process and the output is over-dispersed. Mean age came out at 54.0
against a real 58.8, almost exactly the midpoint of the 18 to 91 range, and mean length of
stay at 77.49 days against a real 4.64. No generated value fell outside the real range, so
this was not an artefact of the clipping step further down. Downstream, that dataset
produced a train-on-synthetic ROC AUC of 0.4903, below the 0.5 of random ranking.

### The result

The sweep trains one model per available continuous encoder on a small subsample, so all of
them finish in a few minutes rather than the hours a full run costs, and reports the numeric
loss alongside the mean and standard deviation of the generated numeric columns.

| Encoder | Numeric loss | Categorical loss | Length of stay, mean (real 4.70) | Length of stay, sd (real 6.84) |
|---|---|---|---|---|
| quantile (plugin default) | 0.9536, not learning | 1.0280 | 59.11 | 55.87 |
| bayesian_gmm | 0.8867, learning | 0.2646 | 4.80 | 5.79 |
| passthrough | failed | | | |

bayesian_gmm is therefore the setting used below. The categorical loss improved as well,
from 1.03 to 0.26, so the encoder was degrading both branches rather than only the
continuous one.

passthrough failed for a reason unrelated to this cohort: the library passes a random_state
argument to every feature encoder while the base class does not accept one. It was included
only as a control, and is redundant now that a working encoder has been found.

### Why this matters

The plugin overrides the library's own encoder default in order to force quantile, and under
that setting the method fails silently on this data. It trains without error, reports a
converged loss, and produces a dataset that looks superficially plausible because the
readmission rate and the value ranges are all correct. Only the distribution statistics
reveal the problem. That silence is itself worth reporting.

In [4]:
import time
import traceback

import torch

print("Importing synthcity (slow on a fresh runtime, up to a few minutes)...", flush=True)
_t = time.time()
from synthcity.plugins import Plugins
from synthcity.plugins.core.dataloader import GenericDataLoader

print(f"  import done in {time.time() - _t:.1f}s", flush=True)

# Deliberately small. This is a yes/no diagnostic about whether the numeric branch
# trains at all, not a quality run, so it is sized to finish in a couple of minutes.
DEBUG_ROWS = 5_000
DEBUG_ITER = 50
DEBUG_TIMESTEPS = 50
DEBUG_BATCH = 512
ENCODERS = ["quantile", "bayesian_gmm", "passthrough"]
NUMERIC_COLS = ["age_at_admission", "length_of_stay_days"]

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"\nSweep on {DEBUG_ROWS:,} rows, {DEBUG_ITER} epochs, device {DEVICE}", flush=True)

debug_df = train_df.sample(n=DEBUG_ROWS, random_state=42).reset_index(drop=True)
real_stats = {c: (debug_df[c].astype(float).mean(), debug_df[c].astype(float).std())
              for c in NUMERIC_COLS}
print("Real subsample:")
for c, (m, s) in real_stats.items():
    print(f"  {c}: mean {m:.2f}, sd {s:.2f}", flush=True)

print("\nBuilding data loader...", flush=True)
debug_loader = GenericDataLoader(debug_df, target_column=TARGET)
print("  loader ready", flush=True)

sweep_rows = []
for encoder in ENCODERS:
    print(f"\n{'=' * 60}\n--- {encoder} ---", flush=True)
    try:
        print("  [1/4] constructing plugin...", flush=True)
        t0 = time.time()
        probe = Plugins().get(
            "ddpm",
            n_iter=DEBUG_ITER,
            batch_size=DEBUG_BATCH,
            num_timesteps=DEBUG_TIMESTEPS,
            device=DEVICE,
            is_classification=True,
            strict=False,
            sampling_patience=1,
            continuous_encoder=encoder,
        )
        print(f"  [2/4] fitting (encoder fit happens first, then the Epoch bar)...", flush=True)
        probe.fit(debug_loader)
        seconds = time.time() - t0
        print(f"  [3/4] fit done in {seconds:.1f}s, generating {DEBUG_ROWS:,} rows...", flush=True)

        history = probe.loss_history
        if history is None:
            history = probe.model.loss_history
        # Average the tail rather than reading one logged point, which is noisy.
        tail = min(5, len(history))
        final_gloss = float(history["gloss"].iloc[-tail:].mean())
        final_mloss = float(history["mloss"].iloc[-tail:].mean())

        sample = probe.generate(count=DEBUG_ROWS).dataframe()
        print(f"  [4/4] generation done", flush=True)

        row = {
            "encoder": encoder,
            "final_gloss": round(final_gloss, 4),
            "final_mloss": round(final_mloss, 4),
            "numeric_learning": final_gloss < 0.95,
            "train_seconds": round(seconds, 1),
        }
        for c in NUMERIC_COLS:
            row[f"{c}_mean"] = round(float(sample[c].astype(float).mean()), 2)
            row[f"{c}_sd"] = round(float(sample[c].astype(float).std()), 2)
        sweep_rows.append(row)

        verdict = "LEARNING" if final_gloss < 0.95 else "NOT LEARNING (trivial solution)"
        print(f"\n  numeric loss {final_gloss:.4f}  ->  {verdict}")
        print(f"  categorical loss {final_mloss:.4f}")
        for c in NUMERIC_COLS:
            rm, rs = real_stats[c]
            print(
                f"  {c}: synthetic mean {row[f'{c}_mean']:>8.2f} sd {row[f'{c}_sd']:>8.2f} "
                f"| real mean {rm:>8.2f} sd {rs:>8.2f}"
            )
    except Exception:
        print(f"  FAILED for encoder={encoder}:")
        traceback.print_exc()
        sweep_rows.append({"encoder": encoder, "final_gloss": None, "numeric_learning": False})
    print(flush=True)

sweep_df = pd.DataFrame(sweep_rows)
out_dir = Path("outputs")
out_dir.mkdir(exist_ok=True)
sweep_df.to_csv(out_dir / "tabddpm_encoder_sweep.csv", index=False)

working = [r["encoder"] for r in sweep_rows if r.get("numeric_learning")]
print("=" * 60)
if working:
    print(f"Encoders whose numeric branch trained: {', '.join(working)}")
    print(f'Set CONTINUOUS_ENCODER = "{working[0]}" in the training cell below.')
else:
    print(
        "No encoder got the numeric loss below 0.95. The numeric branch is not learning\n"
        "regardless of encoding, which points at the library's handling of this data\n"
        "rather than at a tunable setting. Report TabDDPM as a documented\n"
        "implementation failure and keep the three working methods."
    )
sweep_df

Importing synthcity (slow on a fresh runtime, up to a few minutes)...


[transformers] Disabling PyTorch because PyTorch >= 2.4 is required but found 2.2.2
[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


[KeOps] Compiling cuda jit compiler engine ... OK
[pyKeOps] Compiling nvrtc binder for python ... OK
  import done in 31.8s

Sweep on 5,000 rows, 50 epochs, device cuda
Real subsample:
  age_at_admission: mean 59.25, sd 18.98
  length_of_stay_days: mean 4.70, sd 7.44

Building data loader...
  loader ready

--- quantile ---
  [1/4] constructing plugin...


[2026-08-13T14:50:20.991920+0000][5940][CRITICAL] Error importing TabularGoggle: No module named 'dgl'
[2026-08-13T14:50:20.995845+0000][5940][CRITICAL] module disabled: /usr/local/lib/python3.12/dist-packages/synthcity/plugins/generic/plugin_goggle.py


  [2/4] fitting (encoder fit happens first, then the Epoch bar)...


...

[2026-08-13T14:51:01.873647+0000][5940][CRITICAL] module disabled: /usr/local/lib/python3.12/dist-packages/synthcity/plugins/generic/plugin_goggle.py


  [2/4] fitting (encoder fit happens first, then the Epoch bar)...
  FAILED for encoder=passthrough:



Traceback (most recent call last):
  File "/tmp/ipykernel_5940/3220862938.py", line 54, in <cell line: 0>
    probe.fit(debug_loader)
  File "/usr/local/lib/python3.12/dist-packages/pydantic/deprecated/decorator.py", line 56, in wrapper_function
    return vd.call(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pydantic/deprecated/decorator.py", line 151, in call
    return self.execute(m)
           ^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pydantic/deprecated/decorator.py", line 227, in execute
    return self.raw_function(**d, **var_kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/synthcity/plugins/core/plugin.py", line 254, in fit
    output = self._fit(X, *args, **kwargs)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/synthcity/plugins/generic/plugin_ddpm.py", line 221, in _fit
    df = self.encoder.fit_transf

Encoders whose numeric branch trained: quantile
Set CONTINUOUS_ENCODER = "quantile" in the training cell below.


,encoder,final_gloss,final_mloss,numeric_learning,train_seconds,age_at_admission_mean,age_at_admission_sd,length_of_stay_days_mean,length_of_stay_days_sd
0,quantile,0.9336,1.0376,True,13.4,53.29,40.92,81.47,81.39
1,bayesian_gmm,0.9586,0.2544,False,27.0,66.69,33.27,1.75,1.76
2,passthrough,NaN,NaN,False,NaN,NaN,NaN,NaN,NaN


## Hyperparameter search

An earlier round of this project ran every method at library defaults, with the training
budget set by the convergence rule. This section establishes whether TabDDPM performs better under other
settings, so the comparison between methods is not merely a comparison of defaults.

### Method

The search uses successive halving. Every configuration is screened on a small sample at a
reduced budget, and only the strongest few are promoted to a larger sample, the full budget
and repeated seeds. Configurations that look unpromising are therefore abandoned early,
which is where the saving in computation comes from, and the survivors are assessed with
enough repetition that the choice between them is not made on a single noisy run.

Neither library used in this project supports resuming training or validation-based early
stopping at a practical cost, so stopping is applied at the level of the configuration
rather than the epoch. The convergence rule used elsewhere in this project is applied to
each trial's loss curve and reported alongside its score, so a configuration that had not
finished training is visible rather than silently accepted.

Selection uses utility measured on the validation split. The test set is never involved, so
no part of the search can influence the reported results. Fidelity is recorded for every
trial but is not optimised, which means any movement in it is a consequence of selecting for
utility rather than a target of the search. That relationship is itself a finding worth
reporting.

Configurations whose mean scores fall within 0.005 of the best are reported as
indistinguishable, following the same reasoning applied to the differences between methods.
Where several are tied, the simplest should be preferred.

### Parameters varied

The learning rate, weight decay, batch size, the number of denoising steps, and the network itself through `model_params`, which carries the width, depth and dropout. `dim_embed` is varied separately because it sets only the timestep embedding rather than the network capacity. The continuous encoder is held fixed at the value the diagnostic above established on evidence.

### Cost and outputs

Twelve configurations screened, three promoted with three seeds each. This is the most expensive of the four; expect three to four hours.

Three files are written: every individual trial, a summary averaged over seeds, and the
selected configuration as JSON. The training cell below reads the selected
configuration automatically and applies it to the full training split. Nothing here overwrites the saved
synthetic datasets.

In [ ]:
import json
import time

import numpy as np
import pandas as pd
import torch
from pathlib import Path
from scipy.stats import ks_2samp
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# --- Search budget. Reduce these if the search needs to finish sooner. ----------------
SCREEN_ROWS = 25_000     # rows per trial in the screening round
SCREEN_FRACTION = 0.33   # fraction of the full training budget used when screening
PROMOTE_ROWS = 100_000   # rows per trial once a configuration is promoted
PROMOTE_KEEP = 3         # configurations carried into the promotion round
PROMOTE_SEEDS = (0, 1, 2)  # repeats per promoted configuration
TIE_THRESHOLD = 0.005    # ROC AUC difference treated as indistinguishable
FIDELITY_TOLERANCE = 1.5 # a candidate may not worsen KS or TVD beyond this multiple of
                         # the library default, however much utility it gains

NUMERIC_COLS = ["age_at_admission", "length_of_stay_days"]
CATEGORICAL_COLS = [
    "gender", "admission_type", "admission_location", "insurance",
    "marital_status", "race", "language", "had_icu_stay",
]

for _required in ["data/train_fit.parquet", "data/train_val.parquet"]:
    if not Path(_required).exists():
        raise FileNotFoundError(
            f"{_required} not found. Re-run 02_data_preparation.ipynb, which writes the "
            "validation split this search depends on."
        )

fit_df = pd.read_parquet("data/train_fit.parquet")
val_df = pd.read_parquet("data/train_val.parquet")
for _c in fit_df.columns:
    if str(fit_df[_c].dtype) == "Int64":
        fit_df[_c] = fit_df[_c].astype("int64")
        val_df[_c] = val_df[_c].astype("int64")

out_dir = Path("outputs")
out_dir.mkdir(exist_ok=True)

print(f"Fitting split {len(fit_df):,} rows | validation split {len(val_df):,} rows")


def make_classifier() -> Pipeline:
    """The classifier from the main evaluation, so scores are directly comparable."""
    preprocess = ColumnTransformer([
        ("num", StandardScaler(), NUMERIC_COLS),
        ("cat", OneHotEncoder(handle_unknown="ignore"), CATEGORICAL_COLS),
    ])
    return Pipeline([
        ("preprocess", preprocess),
        ("clf", LogisticRegression(max_iter=1000, class_weight="balanced")),
    ])


def utility_on_validation(synthetic: pd.DataFrame) -> float:
    """Train on synthetic, score on the real validation split. The test set is never used."""
    clf = make_classifier()
    clf.fit(synthetic.drop(columns=[TARGET]), synthetic[TARGET])
    proba = clf.predict_proba(val_df.drop(columns=[TARGET]))[:, 1]
    return float(roc_auc_score(val_df[TARGET], proba))


def _tvd(real_col: pd.Series, syn_col: pd.Series) -> float:
    p = real_col.value_counts(normalize=True)
    q = syn_col.value_counts(normalize=True)
    return 0.5 * sum(abs(p.get(c, 0.0) - q.get(c, 0.0)) for c in p.index.union(q.index))


def fidelity_summary(synthetic: pd.DataFrame, real: pd.DataFrame) -> tuple:
    ks = float(np.mean([
        ks_2samp(real[c].astype(float), synthetic[c].astype(float)).statistic
        for c in NUMERIC_COLS
    ]))
    tvd = float(np.mean([
        _tvd(real[c].astype(str), synthetic[c].astype(str)) for c in CATEGORICAL_COLS
    ]))
    return ks, tvd


def converged(loss_series) -> tuple:
    """The stopping rule used elsewhere: mean loss over the final fifth of training
    against the fifth before it. Returns the percentage improvement and a flag."""
    if loss_series is None or len(loss_series) < 10:
        return float("nan"), None
    values = np.asarray(loss_series, dtype=float)
    n = len(values)
    previous = values[int(n * 0.6):int(n * 0.8)].mean()
    final = values[int(n * 0.8):].mean()
    improvement = (previous - final) / abs(previous) * 100
    return float(improvement), bool(improvement <= 1.0)


def set_seed(seed: int) -> None:
    """The sdv synthesizers expose no seed argument, so the global generators are set."""
    np.random.seed(seed)
    torch.manual_seed(seed)


def run_search(space, fit_and_sample, method_slug, full_budget):
    """Successive halving. Every configuration is screened on a small sample at a reduced
    budget; the best few are promoted to a larger sample, a full budget and repeated seeds.
    Unpromising configurations are therefore stopped early, which is where the compute
    saving comes from."""
    screen_sample = fit_df.sample(n=min(SCREEN_ROWS, len(fit_df)), random_state=0).reset_index(drop=True)
    screen_budget = max(1, int(full_budget * SCREEN_FRACTION))

    print(f"\n{'=' * 78}")
    print(f"ROUND 1, screening {len(space)} configurations")
    print(f"{len(screen_sample):,} rows, budget {screen_budget}, one seed each")
    print(f"{'=' * 78}", flush=True)

    screened = []
    for i, (label, config) in enumerate(space, start=1):
        print(f"\n[{i}/{len(space)}] {label}", flush=True)
        t0 = time.time()
        try:
            set_seed(0)
            synthetic, losses = fit_and_sample(config, screen_sample, screen_budget)
            auc = utility_on_validation(synthetic)
            ks, tvd = fidelity_summary(synthetic, screen_sample)
            improvement, is_converged = converged(losses)
            seconds = time.time() - t0
            screened.append({
                "config_label": label, "config": json.dumps(config), "round": "screen",
                "val_roc_auc": auc, "mean_ks": ks, "mean_tvd": tvd,
                "loss_improvement_pct": improvement, "converged": is_converged,
                "seconds": seconds,
            })
            flag = "" if is_converged is None else ("converged" if is_converged else "NOT converged")
            print(f"      ROC AUC {auc:.4f} | KS {ks:.4f} | TVD {tvd:.4f} | {seconds:.0f}s {flag}", flush=True)
        except Exception as exc:  # noqa: BLE001
            print(f"      failed: {type(exc).__name__}: {exc}", flush=True)

    if not screened:
        raise RuntimeError("Every configuration failed during screening.")

    screen_df = pd.DataFrame(screened).sort_values("val_roc_auc", ascending=False)

    # Utility alone is not a sufficient selection criterion. A configuration can raise the
    # downstream score while badly degrading the distributions, which would be a poor
    # outcome for a project whose argument is that these dimensions must be read together.
    # Candidates are therefore restricted to those whose fidelity is no worse than the
    # first configuration in the search space, the library default, by more than the
    # tolerance below. Utility decides the ranking within that set.
    baseline_label = space[0][0]
    baseline = screen_df[screen_df["config_label"] == baseline_label]
    eligible = screen_df
    if not baseline.empty:
        base_ks = float(baseline.iloc[0]["mean_ks"])
        base_tvd = float(baseline.iloc[0]["mean_tvd"])
        limit_ks = base_ks * FIDELITY_TOLERANCE + 1e-6
        limit_tvd = base_tvd * FIDELITY_TOLERANCE + 1e-6
        eligible = screen_df[(screen_df["mean_ks"] <= limit_ks)
                             & (screen_df["mean_tvd"] <= limit_tvd)]
        excluded = screen_df[~screen_df["config_label"].isin(eligible["config_label"])]
        print()
        print(f"Fidelity guardrail: KS <= {limit_ks:.4f}, TVD <= {limit_tvd:.4f}")
        print(f"  ({FIDELITY_TOLERANCE}x the baseline configuration '{baseline_label}')")
        if len(excluded):
            print(f"  Excluded {len(excluded)} configuration(s) that raised utility at the "
                  f"cost of fidelity:")
            for _, r in excluded.iterrows():
                print(f"    {r['config_label']:<42} ROC AUC {r['val_roc_auc']:.4f}  "
                      f"KS {r['mean_ks']:.4f}  TVD {r['mean_tvd']:.4f}")
        if eligible.empty:
            print("  No configuration met the guardrail, so it has been relaxed for this run.")
            eligible = screen_df

    promoted = eligible.head(PROMOTE_KEEP)

    print(f"\n{'=' * 78}")
    print(f"ROUND 2, promoting the top {len(promoted)} of {len(screen_df)}")
    print(f"{min(PROMOTE_ROWS, len(fit_df)):,} rows, budget {full_budget}, "
          f"{len(PROMOTE_SEEDS)} seeds each")
    print(f"{'=' * 78}", flush=True)

    promote_sample = fit_df.sample(n=min(PROMOTE_ROWS, len(fit_df)), random_state=0).reset_index(drop=True)
    promoted_rows = []
    for i, row in enumerate(promoted.itertuples(index=False), start=1):
        config = json.loads(row.config)
        print(f"\n[{i}/{len(promoted)}] {row.config_label}", flush=True)
        for seed in PROMOTE_SEEDS:
            t0 = time.time()
            try:
                set_seed(seed)
                synthetic, losses = fit_and_sample(config, promote_sample, full_budget, seed=seed)
                auc = utility_on_validation(synthetic)
                ks, tvd = fidelity_summary(synthetic, promote_sample)
                improvement, is_converged = converged(losses)
                promoted_rows.append({
                    "config_label": row.config_label, "config": row.config, "round": "promote",
                    "seed": seed, "val_roc_auc": auc, "mean_ks": ks, "mean_tvd": tvd,
                    "loss_improvement_pct": improvement, "converged": is_converged,
                    "seconds": time.time() - t0,
                })
                print(f"      seed {seed}: ROC AUC {auc:.4f} | KS {ks:.4f} | TVD {tvd:.4f}", flush=True)
            except Exception as exc:  # noqa: BLE001
                print(f"      seed {seed} failed: {type(exc).__name__}: {exc}", flush=True)

    promote_df = pd.DataFrame(promoted_rows)
    summary = (
        promote_df.groupby(["config_label", "config"])
        .agg(mean_roc_auc=("val_roc_auc", "mean"), sd_roc_auc=("val_roc_auc", "std"),
             mean_ks=("mean_ks", "mean"), mean_tvd=("mean_tvd", "mean"),
             runs=("val_roc_auc", "size"))
        .reset_index().sort_values("mean_roc_auc", ascending=False)
    )

    pd.concat([screen_df, promote_df], ignore_index=True).to_csv(
        out_dir / f"tuning_{method_slug}_trials.csv", index=False)
    summary.to_csv(out_dir / f"tuning_{method_slug}_summary.csv", index=False)

    print(f"\n{'=' * 78}")
    print("PROMOTION RESULTS, mean over seeds")
    print(f"{'=' * 78}")
    for _, r in summary.iterrows():
        sd = 0.0 if pd.isna(r["sd_roc_auc"]) else r["sd_roc_auc"]
        print(f"  {r['config_label']:<46} {r['mean_roc_auc']:.4f} +/- {sd:.4f}  "
              f"KS {r['mean_ks']:.4f}  TVD {r['mean_tvd']:.4f}  (n={r['runs']})")

    # Utility decides the ranking, but configurations within TIE_THRESHOLD of the leader
    # are not separable on this evidence. Rather than taking whichever happened to score
    # highest, the tie is broken on fidelity: lower distributional error first, and the
    # library default preferred over an equally faithful alternative.
    leader = summary.iloc[0]["mean_roc_auc"]
    tied = summary[summary["mean_roc_auc"] >= leader - TIE_THRESHOLD].copy()
    tied["is_default"] = tied["config"] == "{}"
    tied = tied.sort_values(
        ["mean_ks", "mean_tvd", "is_default"], ascending=[True, True, False]
    )
    best = tied.iloc[0]

    checkpoint = {
        "method": method_slug,
        "selected_config": json.loads(best["config"]),
        "selected_label": best["config_label"],
        "mean_val_roc_auc": float(best["mean_roc_auc"]),
        "sd_val_roc_auc": None if pd.isna(best["sd_roc_auc"]) else float(best["sd_roc_auc"]),
        "seeds": list(PROMOTE_SEEDS),
        "screen_rows": int(min(SCREEN_ROWS, len(fit_df))),
        "promote_rows": int(min(PROMOTE_ROWS, len(fit_df))),
        "full_budget": full_budget,
        "tied_within_threshold": tied["config_label"].tolist(),
    }
    with open(out_dir / f"tuning_{method_slug}_selected.json", "w", encoding="utf-8") as fh:
        json.dump(checkpoint, fh, indent=2)

    print()
    print(f"Selected: {best['config_label']}")
    if len(tied) > 1:
        print(f"{len(tied)} configurations fell within {TIE_THRESHOLD} ROC AUC of the leader "
              f"and are not separable on utility:")
        for _, r in tied.iterrows():
            print(f"    {r['config_label']:<44} ROC AUC {r['mean_roc_auc']:.4f}  "
                  f"KS {r['mean_ks']:.4f}  TVD {r['mean_tvd']:.4f}")
        print("The tie was broken on fidelity, preferring the lowest distributional error.")
        print("Report the tie rather than presenting the winner as clearly better.")
    print(f"\nSaved outputs/tuning_{method_slug}_selected.json for the final run.")
    return summary


METHOD_SLUG = "tabddpm"
FULL_BUDGET = 1000  # iterations, matching the converged budget used for the reported run

import functools

from synthcity.plugins import Plugins
from synthcity.plugins.core.dataloader import GenericDataLoader

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# model_params carries the network capacity; dim_embed is only the timestep embedding, so
# both are varied. The continuous encoder is held at bayesian_gmm because the diagnostic
# earlier in this notebook settled it on evidence. num_timesteps is included because it
# governs sample quality directly and was reduced during earlier exploratory work.
SEARCH_SPACE = [
    ("reported settings", {}),
    ("lr=0.001", {"lr": 0.001}),
    ("lr=0.005", {"lr": 0.005}),
    ("wide network 512", {"model_params": {"n_units_hidden": 512, "n_layers_hidden": 3, "dropout": 0.0}}),
    ("deep network 5 layers", {"model_params": {"n_units_hidden": 256, "n_layers_hidden": 5, "dropout": 0.0}}),
    ("dropout 0.1", {"model_params": {"n_units_hidden": 256, "n_layers_hidden": 3, "dropout": 0.1}}),
    ("dim_embed=256", {"dim_embed": 256}),
    ("timesteps=500", {"num_timesteps": 500}),
    ("timesteps=2000", {"num_timesteps": 2000}),
    ("batch_size=1024", {"batch_size": 1024}),
    ("weight_decay=1e-3", {"weight_decay": 1e-3}),
    ("lr=0.001 + wide 512", {"lr": 0.001, "model_params": {
        "n_units_hidden": 512, "n_layers_hidden": 3, "dropout": 0.0}}),
]


def fit_and_sample(config, data, budget, seed=0):
    # The batch size is scaled to the sample rather than fixed, so that a screening trial
    # is a smaller version of the full run rather than a different training regime. Held
    # at 4,096 the screening sample of 25,000 rows would give six gradient updates per
    # epoch against 105 on the full data, which was enough to diverge to NaN even for the
    # configuration that trains successfully at full scale. This keeps roughly fifty
    # updates per epoch at any sample size. An explicit batch_size in a configuration
    # still overrides it, since the update below is applied afterwards.
    settings = {
        "n_iter": budget,
        "batch_size": int(max(256, min(4096, len(data) // 50))),
        "num_timesteps": 1000,
        "device": DEVICE,
        "continuous_encoder": "bayesian_gmm",
        "is_classification": True,
        "strict": False,
        "sampling_patience": 1,
        "random_state": seed,
    }
    settings.update(config)

    print(f"      batch_size={settings['batch_size']}, "
          f"{max(1, len(data) // settings['batch_size'])} updates per epoch", flush=True)

    model = Plugins().get("ddpm", **settings)
    loader = GenericDataLoader(data, target_column=TARGET)
    model.fit(loader)

    diffusion = model.model.diffusion
    if not getattr(diffusion, "_batch_size_patched", False):
        diffusion.sample_all = functools.partial(diffusion.sample_all, max_batch_size=50_000)
        diffusion._batch_size_patched = True

    # Check for divergence before generating. A run whose loss has become non-finite
    # cannot produce usable data, and generation is the expensive half of a trial.
    losses = None
    history = model.loss_history if model.loss_history is not None else model.model.loss_history
    if history is not None:
        losses = history["loss"].to_numpy()
        if not np.isfinite(losses[-5:]).all():
            raise ValueError("training diverged, loss became non-finite")

    synthetic = model.generate(count=len(data)).dataframe()
    synthetic = synthetic[data.columns.tolist()]
    synthetic[TARGET] = synthetic[TARGET].astype(int)

    if not np.isfinite(synthetic[NUMERIC_COLS].to_numpy(dtype=float)).all():
        raise ValueError("generated data contains non-finite numeric values")

    return synthetic, losses

summary = run_search(SEARCH_SPACE, fit_and_sample, METHOD_SLUG, FULL_BUDGET)
summary


Fitting split 342,342 rows | validation split 85,801 rows

ROUND 1, screening 12 configurations
25,000 rows, budget 330, one seed each

[1/12] reported settings


[2026-08-13T14:51:07.456533+0000][5940][CRITICAL] module disabled: /usr/local/lib/python3.12/dist-packages/synthcity/plugins/generic/plugin_goggle.py
Epoch: 100%|██████████| 330/330 [03:11<00:00,  1.72it/s, loss=nan]


      failed: ValueError: found NaNs in sample

[2/12] lr=0.001


[2026-08-13T14:54:51.517900+0000][5940][CRITICAL] module disabled: /usr/local/lib/python3.12/dist-packages/synthcity/plugins/generic/plugin_goggle.py
Epoch:  57%|█████▋    | 189/330 [01:49<01:14,  1.89it/s, loss=nan]Exception ignored in: <function _xla_gc_callback at 0x7b2bacbf22a0>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/jax/_src/lib/__init__.py", line 127, in _xla_gc_callback
    def _xla_gc_callback(*args):
    
KeyboardInterrupt: 
Epoch:  65%|██████▍   | 214/330 [02:04<01:09,  1.66it/s, loss=nan]

: 

: 

Epoch:  65%|██████▌   | 215/330 [02:04<01:06,  1.72it/s, loss=nan]


KeyboardInterrupt: 

## Generating both datasets

Two datasets are produced from this method, not one. The first uses the library defaults,
the second the configuration chosen by the search above. Both are generated in this
notebook so that a single run evidences both experiments, and so the comparison between
them is made on identical data, an identical split and identical evaluation code.

Each is written under its own filename and recorded as its own row in the shared timing
log, tagged with the configuration and the hardware. The evaluation notebook picks up
whichever datasets exist, so it reports both without further intervention.

The two runs are in separate cells deliberately. Training is the expensive part, and a
failure in the second should not discard the first.

In [ ]:
# Training settings shared by both experiments. The search may override any of them, but
# these are the values the convergence work established.
N_ITER = 1000              # passes over the data
BATCH_SIZE = 4096          # records per training step
NUM_TIMESTEPS = 1000       # denoising steps, the library default
CONTINUOUS_ENCODER = "bayesian_gmm"  # selected by the diagnostic earlier in this notebook
CHUNK_SIZE = 50_000        # rows per generation call, and the internal sampling batch

METHOD_NAME = "TabDDPM"
METHOD_SLUG = "tabddpm"

import json as _json
import time

import numpy as np
import pandas as pd
from pathlib import Path

out_dir = Path("outputs")
out_dir.mkdir(exist_ok=True)
fig_dir = Path("figures")
fig_dir.mkdir(exist_ok=True)

try:
    import torch as _torch
    GPU_NAME = _torch.cuda.get_device_name(0) if _torch.cuda.is_available() else "CPU"
except Exception:  # noqa: BLE001
    GPU_NAME = "unknown"

# The tuned configuration comes from the search above. If it has not been run, only the
# default experiment is possible and the tuned cell will say so rather than failing.
TUNED_CONFIG = None
TUNED_LABEL = None
_selected = Path(f"outputs/tuning_{METHOD_SLUG}_selected.json")
if _selected.exists():
    with open(_selected, encoding="utf-8") as _fh:
        _payload = _json.load(_fh)
    TUNED_CONFIG = _payload["selected_config"]
    TUNED_LABEL = _payload["selected_label"]
    print(f"Tuned configuration available: {TUNED_LABEL}")
    print(f"  {TUNED_CONFIG}")
else:
    print(f"No search result at {_selected}. Only the default experiment can be run.")

results = {}


def convergence(losses, n_label):
    """The stopping rule used throughout: mean loss over the final fifth of training
    against the fifth before it."""
    if losses is None or len(losses) < 10:
        return None, None
    values = np.asarray(losses, dtype=float)
    n = len(values)
    previous = values[int(n * 0.6):int(n * 0.8)].mean()
    final = values[int(n * 0.8):].mean()
    improvement = (previous - final) / abs(previous) * 100
    print(f"  convergence: {previous:.4f} -> {final:.4f}, {improvement:+.2f}% across the "
          f"final fifth ({'flat' if improvement <= 1.0 else 'still improving'}, {n_label})")
    return float(improvement), bool(improvement <= 1.0)


def sanity(synthetic):
    checks = pd.DataFrame({
        "statistic": ["Readmission rate", "Mean age", "Mean length of stay (days)"],
        "real": [
            round(train_df[TARGET].mean(), 4),
            round(train_df["age_at_admission"].astype(float).mean(), 1),
            round(train_df["length_of_stay_days"].mean(), 2),
        ],
        "synthetic": [
            round(synthetic[TARGET].mean(), 4),
            round(synthetic["age_at_admission"].astype(float).mean(), 1),
            round(synthetic["length_of_stay_days"].mean(), 2),
        ],
    })
    print(checks.to_string(index=False))
    return checks


def run_experiment(label, config, slug):
    """Fit, generate, save and summarise one configuration."""
    print("=" * 78)
    print(f"{METHOD_NAME}: {label}")
    print("=" * 78, flush=True)

    synthetic, train_seconds, generate_seconds, losses = fit_and_generate(config, slug)

    path = f"data/synthetic_{METHOD_SLUG}_{slug}.parquet"
    synthetic.to_parquet(path, index=False)

    improvement, converged = convergence(losses, label)
    if losses is not None:
        try:
            frame = pd.DataFrame({"loss": np.asarray(losses, dtype=float)})
            frame.to_csv(out_dir / f"loss_history_{METHOD_SLUG}_{slug}.csv", index=False)
        except Exception as exc:  # noqa: BLE001
            print(f"  could not write the loss history: {type(exc).__name__}")

    print(f"  training {train_seconds:.1f}s, generation {generate_seconds:.1f}s, "
          f"{len(synthetic):,} rows -> {path}")
    sanity(synthetic)

    record = {
        "method": METHOD_NAME,
        "experiment": slug,
        "config": label,
        "rows_generated": len(synthetic),
        "train_seconds": round(train_seconds, 1),
        "generate_seconds": round(generate_seconds, 1),
        "gpu": GPU_NAME,
        "loss_improvement_pct": improvement,
        "converged": converged,
    }
    results[slug] = record
    return record


def fit_and_generate(config, slug=''):
    import functools

    import torch

    from synthcity.plugins import Plugins
    from synthcity.plugins.core.dataloader import GenericDataLoader

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    settings = {
        "n_iter": N_ITER,
        "batch_size": BATCH_SIZE,
        "num_timesteps": NUM_TIMESTEPS,
        "device": device,
        "continuous_encoder": CONTINUOUS_ENCODER,
        # Treat readmission as the class label so generation is conditioned on the outcome.
        "is_classification": True,
        # These disable the library's internal resampling loop, which otherwise regenerates
        # the whole dataset until enough rows survive its own filters.
        "strict": False,
        "sampling_patience": 1,
    }
    settings.update(config)

    model = Plugins().get("ddpm", **settings)
    loader = GenericDataLoader(train_df, target_column=TARGET)

    t0 = time.time()
    model.fit(loader)
    train_seconds = time.time() - t0

    # The library samples in fixed batches of 2,000 with no progress reporting, which
    # leaves the accelerator idle and makes a long generation indistinguishable from a
    # hang. Raising the batch cuts the number of sequential steps proportionally.
    diffusion = model.model.diffusion
    if not getattr(diffusion, "_batch_size_patched", False):
        diffusion.sample_all = functools.partial(diffusion.sample_all, max_batch_size=CHUNK_SIZE)
        diffusion._batch_size_patched = True

    t0 = time.time()
    pieces = []
    for start in range(0, len(train_df), CHUNK_SIZE):
        n = min(CHUNK_SIZE, len(train_df) - start)
        pieces.append(model.generate(count=n).dataframe())
        print(f"    {start + n:,}/{len(train_df):,} rows", flush=True)
    generate_seconds = time.time() - t0

    synthetic = pd.concat(pieces, ignore_index=True)[train_df.columns.tolist()]
    synthetic[TARGET] = synthetic[TARGET].astype(int)

    # The sdv methods keep generated values inside the observed range; this library does
    # not once its own filtering is disabled, so the same bound is applied and reported.
    clipped = []
    for col in ["age_at_admission", "length_of_stay_days"]:
        lo = float(train_df[col].astype(float).min())
        hi = float(train_df[col].astype(float).max())
        values = synthetic[col].astype(float)
        n_out = int(((values < lo) | (values > hi)).sum())
        synthetic[col] = values.clip(lo, hi)
        clipped.append({"column": col, "n_clipped": n_out,
                        "pct_clipped": round(n_out / len(synthetic) * 100, 4)})
        print(f"    {col}: {n_out:,} values ({n_out / len(synthetic):.2%}) clipped to "
              f"[{lo:.2f}, {hi:.2f}]")
    pd.DataFrame(clipped).to_csv(out_dir / f"clip_report_tabddpm_{slug}.csv", index=False)

    losses = None
    history = model.loss_history if model.loss_history is not None else model.model.loss_history
    if history is not None:
        losses = history["loss"].to_numpy()
    return synthetic, train_seconds, generate_seconds, losses


In [ ]:
run_experiment("library defaults", {}, "default")

In [ ]:
if TUNED_CONFIG is None:
    print("No tuned configuration is available. Run the search above first.")
else:
    run_experiment(TUNED_LABEL, TUNED_CONFIG, "tuned")

In [ ]:
# Append both runs to the shared log, keyed on method and experiment so neither
# replaces the other.
log_path = out_dir / "generation_log.csv"
entries = pd.DataFrame(list(results.values()))

if log_path.exists():
    log = pd.read_csv(log_path)
    if "experiment" in log.columns:
        # Replace this method's rows for the experiments just run, and also drop any row
        # for it that predates the two-experiment structure. Those were written from a
        # superseded split and would otherwise persist unnoticed into the cost table.
        stale = log["method"] == METHOD_NAME
        stale &= log["experiment"].isin(entries["experiment"]) | log["experiment"].isna()
        log = log[~stale]
    else:
        log = log[log["method"] != METHOD_NAME]
    log = pd.concat([log, entries], ignore_index=True)
else:
    log = entries

try:
    log.to_csv(log_path, index=False)
except Exception as exc:  # noqa: BLE001
    import csv as _csv
    with open(log_path, "w", newline="", encoding="utf-8") as fh:
        writer = _csv.writer(fh)
        writer.writerow([str(c) for c in log.columns])
        writer.writerows(log.itertuples(index=False, name=None))
    print(f"wrote the log via the standard library ({type(exc).__name__})")

print()
print("=" * 78)
print(f"{METHOD_NAME}: both experiments complete")
print("=" * 78)
print(entries[["experiment", "config", "train_seconds", "generate_seconds", "gpu"]].to_string(index=False))
if len(results) == 2:
    d, t = results["default"], results["tuned"]
    delta = t["train_seconds"] + t["generate_seconds"] - d["train_seconds"] - d["generate_seconds"]
    print(f"\nThe tuned configuration cost {delta:+.1f}s in total compute.")
    print("Whether it bought anything is decided in 07_evaluation.ipynb, not here.")
log
